In [85]:
## Necessary imports

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.stats.multitest import multipletests

In [ ]:

"""
Non-parametric comparison of AUC between stress and stim conditions across multiple timepoints
with repeated measurements per animal.

Workflow (per timepoint):
1) Build table (animal, condition, auc)
2) Collapse repeats within each animal+condition using median 
3) Compare conditions using an animal-level permutation test (two-sided)
   on the difference in medians between conditions
4) Report effect sizes:
   - Hodges–Lehmann estimator (median of all pairwise differences stim - stress)
   - Cliff's delta
   with bootstrap 95% CI for Hodges–Lehmann
5) Correct p-values across timepoints with Benjamini–Hochberg (FDR)

Run:
    python nonparam_auc_analysis.py
"""

from __future__ import annotations

import numpy as np
import pandas as pd



#animal IDs (repeats per animal in stim condition)
stim_animal = [
    "G7", "G7", "G8", "G8", "G8", "G8", "G9", "G9", "G9", "G9",
    "G28", "G28", "G28", "G29", "G29", "G29", "G37", "G37", "G37", "G37"
]
#animal IDs (repeats per animal in stress condition)
stress_animal = ["G7", "G8", "G9", "G28", "G29", "G37", "G37"]

# AUC values per timepoint
AUC_stim_30   = [-3.2941, -0.8233, 1.4743, -1.9491, 2.5396, 1.0696, -0.6789, -0.4772, -0.6625, -2.1106,
                 -2.6996, 10.156, 1.2475, 4.7087, -1.5431, 3.906, 1.4121, -2.2647, -1.1277, -3.9984]
AUC_stress_30 = [3.9668, 4.511, 2.5152, 60.6365, 23.0652, 22.4978, 41.8574]

AUC_stim_60   = [-3.0155, -1.267, 1.6671, -2.3267, 0.952, -0.4225, -2.2116, 1.5772, -0.1854, -1.7976,
                 -1.8799, 14.8400, -2.0294, 4.1857, -0.9583, 1.1032, 0.0982, -2.7538, -0.5514, -3.5729]
AUC_stress_60 = [4.7454, 2.3182, 0.4937, 50.6340, 21.3518, 26.5827, 51.8101]

AUC_stim_90   = [-1.2230, -1.1646, 1.5550, -1.1195, 1.0739, -0.7815, -2.6634, 1.4759, -0.1702, -3.1705,
                 -0.7590, 13.0363, -2.0926, 3.4858, 1.1061, 2.0930, -2.0750, -4.4860, -1.3451, -3.1809]
AUC_stress_90 = [4.2257, 1.6342, -1.6458, 39.6093, 17.7396, 24.2487, 50.9404]

AUC_stim_120   = [-0.6409, -0.7567, 0.7624, -0.5384, 0.7329, -0.8672, 0.0113, 1.1853, 0.6831, -0.7784,
                  -0.8053, 5.1225, -0.5467, 1.7969, -0.0616, 0.3441, -1.2354, -2.4234, -0.7771, -1.499]
AUC_stress_120 = [1.0498, -1.5431, -0.7729, 17.6906, 11.2251, 6.8057, 20.8275]


# -----------------------------
# Helpers
# -----------------------------
def animal_level_median(stim_animals, stim_vals, stress_animals, stress_vals) -> pd.DataFrame:
    """Return animal-level medians per condition in long-form."""
    stim_df = pd.DataFrame({"animal": stim_animals, "auc": stim_vals, "condition": "stim"})
    stress_df = pd.DataFrame({"animal": stress_animals, "auc": stress_vals, "condition": "stress"})
    df = pd.concat([stim_df, stress_df], ignore_index=True)

    # Collapse repeats within each animal & condition (robust)
    collapsed = (
        df.groupby(["animal", "condition"], as_index=False)["auc"]
          .median()
    )
    return collapsed


def permutation_test_diff_of_medians(x: np.ndarray, y: np.ndarray, n_perm: int = 10000, seed: int = 0) -> float:
    """
    Two-sided permutation test for difference in medians: median(x) - median(y).
    Operates at animal level (x and y are already animal-level summaries).
    """
    rng = np.random.default_rng(seed)

    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    observed = np.median(x) - np.median(y)

    pooled = np.concatenate([x, y])
    nx = len(x)

    count = 0
    for _ in range(n_perm):
        rng.shuffle(pooled)
        x_perm = pooled[:nx]
        y_perm = pooled[nx:]
        stat = np.median(x_perm) - np.median(y_perm)
        if abs(stat) >= abs(observed):
            count += 1

    # add 1 to numerator+denominator for a conservative finite-sample p-value
    return (count + 1) / (n_perm + 1)


def hodges_lehmann(x: np.ndarray, y: np.ndarray) -> float:
    """Hodges–Lehmann estimator: median of all pairwise differences x_i - y_j."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    diffs = (x[:, None] - y[None, :]).ravel()
    return float(np.median(diffs))


def bootstrap_ci_hl(x: np.ndarray, y: np.ndarray, n_boot: int = 10000, seed: int = 0) -> tuple[float, float]:
    """Bootstrap 95% CI for HL estimator (resample within each group)."""
    rng = np.random.default_rng(seed)
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    est = np.empty(n_boot, dtype=float)
    for i in range(n_boot):
        xb = rng.choice(x, size=len(x), replace=True)
        yb = rng.choice(y, size=len(y), replace=True)
        est[i] = hodges_lehmann(xb, yb)

    lo, hi = np.percentile(est, [2.5, 97.5])
    return float(lo), float(hi)


def cliffs_delta(x: np.ndarray, y: np.ndarray) -> float:
    """Cliff's delta: (P(x>y) - P(x<y))."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    # pairwise comparisons
    gt = (x[:, None] > y[None, :]).sum()
    lt = (x[:, None] < y[None, :]).sum()
    n = x.size * y.size
    return float((gt - lt) / n)



def analyze_timepoint(label: str, stim_vals, stress_vals, n_perm=20000, n_boot=20000, seed=0) -> dict:
    collapsed = animal_level_median(stim_animal, stim_vals, stress_animal, stress_vals)

    stim_animals = collapsed.loc[collapsed["condition"] == "stim", "animal"].nunique()
    stress_animals = collapsed.loc[collapsed["condition"] == "stress", "animal"].nunique()

    x = collapsed.loc[collapsed["condition"] == "stim", "auc"].to_numpy()
    y = collapsed.loc[collapsed["condition"] == "stress", "auc"].to_numpy()

    # Primary test
    p_perm = permutation_test_diff_of_medians(x, y, n_perm=n_perm, seed=seed)

    # Effect sizes
    hl = hodges_lehmann(x, y)  # stim - stress
    hl_lo, hl_hi = bootstrap_ci_hl(x, y, n_boot=n_boot, seed=seed + 1)
    cd = cliffs_delta(x, y)

    # Descriptives (animal-level)
    med_stim = float(np.median(x))
    med_stress = float(np.median(y))

    return {
        "timepoint": label,
        "n_animals_stim": stim_animals,
        "n_animals_stress": stress_animals,
        "median_stim": med_stim,
        "median_stress": med_stress,
        "test": "animal-level permutation (diff of medians)",
        "p_perm": p_perm,
        "HL(stim-stress)": hl,
        "HL_CI95_low": hl_lo,
        "HL_CI95_high": hl_hi,
        "Cliffs_delta": cd,
    }




In [4]:
# You can tune these for speed vs precision:
N_PERM = 10000
N_BOOT = 10000
SEED = 123

results = [
    analyze_timepoint("30s",  AUC_stim_30,  AUC_stress_30,  n_perm=N_PERM, n_boot=N_BOOT, seed=SEED + 30),
    analyze_timepoint("60s",  AUC_stim_60,  AUC_stress_60,  n_perm=N_PERM, n_boot=N_BOOT, seed=SEED + 60),
    analyze_timepoint("90s",  AUC_stim_90,  AUC_stress_90,  n_perm=N_PERM, n_boot=N_BOOT, seed=SEED + 90),
    analyze_timepoint("120s", AUC_stim_120, AUC_stress_120, n_perm=N_PERM, n_boot=N_BOOT, seed=SEED + 120),
]

df = pd.DataFrame(results)





# Nicer formatting for console
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 50)

cols = [
    "timepoint",
    "n_animals_stim", "n_animals_stress",
    "median_stim", "median_stress",
    "HL(stim-stress)", "HL_CI95_low", "HL_CI95_high",
    "Cliffs_delta",
    "p_perm",
]
print(df[cols].to_string(index=False, float_format=lambda x: f"{x:0.4g}"))


timepoint  n_animals_stim  n_animals_stress  median_stim  median_stress  HL(stim-stress)  HL_CI95_low  HL_CI95_high  Cliffs_delta  p_perm
      30s               6                 6       0.2884          13.79           -12.86       -45.48        -2.979       -0.9444  0.0132
      60s               6                 6       -1.322          13.05           -13.57       -45.43        -2.564       -0.9444  0.0121
      90s               6                 6      -0.9764          10.98           -11.25        -38.9       -0.9822       -0.6667  0.0421
     120s               6                 6      -0.2247          6.137           -6.647       -15.93        0.8443       -0.3889 0.09709
